Environment & Load Model SFT

In [1]:
!pip install -q --upgrade unsloth unsloth_zoo datasets wandb

import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "PGABL-legal-chatbot-grpo"
os.environ["WANDB_LOG_MODEL"] = "false"

from unsloth import FastLanguageModel   # import unsloth SEBELUM trl
try:
    from unsloth import PatchFastRL
    PatchFastRL("GRPO", FastLanguageModel)
except Exception as e:
    print("PatchFastRL dilewati (versi Unsloth baru sudah patch otomatis):", e)

import torch, wandb
wandb.login(key=os.environ["WANDB_API_KEY"])

SEED = 3407
SFT_REPO = "putraharifin/llama-3.2-3b-legal-id-PGABL-putraharifin25"
max_seq_length = 1024   # prompt (<=256) + completion (<=384) jauh di bawah batas ini
lora_rank = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_REPO,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
    fast_inference=False,   # tanpa vLLM: lebih stabil di T4/P100
    token=os.environ["HF_TOKEN"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_rank * 2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()
print(f"VRAM terpakai: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.6/28.6 MB 68.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 81.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 82.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.5/176.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.1/135.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: putraharifin (putraharifin-universitas-sumatera-utara) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


==((====))==  Unsloth 2026.10.9: Fast Llama patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load putraharifin/llama-3.2-3b-legal-id-PGABL-putraharifin25 as a legacy tokenizer.
Unsloth 2026.10.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511
VRAM terpakai: 2.22 GB


4 Reward Functions

In [2]:
import re

# ---------- Helper ----------
def get_text(completion):
    """TRL mengirim completion berupa str atau list[{'role','content'}]."""
    if isinstance(completion, str):
        return completion
    return completion[0]["content"]

def get_final_answer(text):
    """Teks setelah </think> (jawaban akhir). Kosong jika tag penutup tidak ada."""
    if "</think>" not in text:
        return ""
    return text.split("</think>", 1)[1].strip()

ID_WORDS = {"yang", "dan", "di", "ke", "dari", "untuk", "dengan", "pada", "adalah", "ini", "itu",
            "atau", "tidak", "akan", "dalam", "juga", "oleh", "sebagai", "karena", "dapat", "ada",
            "saya", "anda", "kamu", "mereka", "kami", "bisa", "harus", "telah", "sudah", "lebih",
            "jika", "maka", "serta", "bagi", "agar", "setiap", "tersebut", "secara", "para", "berhak"}
EN_WORDS = {"the", "and", "is", "are", "of", "to", "that", "with", "for", "this", "you", "your",
            "can", "will", "have", "has", "not", "it", "be", "was", "were", "from", "on", "at",
            "an", "or", "if", "which", "they", "their", "we", "our", "what", "when", "how", "there"}
STOPWORDS_KEYWORD = ID_WORDS | {"seperti", "yaitu", "yakni", "antara", "sangat", "banyak", "lain",
                                "dapat", "hal", "lebih", "baik", "satu", "dua", "tiga", "juga"}

# ---------- 1. format_reward_func ----------
def format_reward_func(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        text = get_text(c)
        n_open, n_close = text.count("<think>"), text.count("</think>")
        score = 0.0
        if n_open >= 1:
            score += 0.2                      # membuka dengan <think>
        if n_close >= 1:
            score += 0.3                      # menutup dengan </think>
        perfect = re.match(r"^\s*<think>\s*(.+?)\s*</think>\s*(\S.*)$", text, flags=re.DOTALL)
        if perfect and n_open == 1 and n_close == 1:
            score = 1.0                       # format sempurna (maksimal +1.0)
        if n_open > 1 or n_close > 1:
            score -= 0.5                      # penalti tag berulang / halusinasi
        rewards.append(score)
    return rewards

# ---------- 2. reasoning_length_reward ----------
def reasoning_length_reward(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        text = get_text(c)
        # toleran terhadap reasoning yang terpotong max_completion_length (tanpa </think>)
        m = re.search(r"<think>(.*?)(?:</think>|$)", text, flags=re.DOTALL)
        if not m or not m.group(1).strip():
            rewards.append(0.0)
            continue
        n = len(m.group(1).strip())
        if n < 50:
            rewards.append(0.2)
        elif n < 200:
            rewards.append(0.5)
        else:
            rewards.append(1.0)
    return rewards

# ---------- 3. correctness_reward ----------
def normalize(s):
    return re.sub(r"[^a-z0-9\s]", " ", s.lower())

def extract_keywords(ground_truth, max_kw=12):
    words = [w for w in normalize(ground_truth).split() if len(w) >= 4 and w not in STOPWORDS_KEYWORD]
    uniq = list(dict.fromkeys(words))
    uniq.sort(key=len, reverse=True)          # kata panjang cenderung lebih spesifik
    return uniq[:max_kw]

def correctness_reward(prompts, completions, answer, **kwargs):
    rewards = []
    for c, gt in zip(completions, answer):
        final = normalize(get_final_answer(get_text(c)))
        if not final.strip():
            rewards.append(0.0)
            continue
        gt_norm = " ".join(normalize(gt).split())
        if gt_norm and gt_norm in " ".join(final.split()):
            rewards.append(1.0)               # ground truth muncul utuh
            continue
        kws = extract_keywords(gt)
        if not kws:
            rewards.append(0.0)
            continue
        hit = sum(1 for k in kws if k in final)
        rewards.append(1.0 if hit / len(kws) >= 0.4 else 0.0)   # >=40% kata kunci ground truth
    return rewards

# ---------- 4. language_reward_func ----------
def language_reward_func(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        text = get_text(c)
        final = get_final_answer(text)
        if not final:                         # fallback: seluruh teks tanpa tag
            final = re.sub(r"</?think>", " ", text)
        words = re.findall(r"[a-zA-Z]+", final.lower())
        if len(words) < 3:
            rewards.append(0.0)
            continue
        n_id = sum(w in ID_WORDS for w in words)
        n_en = sum(w in EN_WORDS for w in words)
        if n_en > n_id and n_en / len(words) > 0.10:
            rewards.append(-0.5)              # menjawab bahasa Inggris
        elif n_en == 0 and n_id / len(words) >= 0.10:
            rewards.append(1.0)               # murni Bahasa Indonesia
        else:
            rewards.append(0.0)
    return rewards

# ---------- Sanity check ----------
_t = ["<think>\nStaf admin termasuk pekerja non-manajerial. Berdasarkan PP 35/2021, kerja melebihi waktu kerja normal wajib dibayar upah lembur, sehingga 3 jam lembur harus dibayar sesuai ketentuan perhitungan upah.\n</think>\nYa, Anda berhak mendapat upah lembur untuk 3 jam tersebut."]
print("format:", format_reward_func(None, _t), "| length:", reasoning_length_reward(None, _t),
      "| correct:", correctness_reward(None, _t, answer=["Anda berhak mendapat upah lembur"]),
      "| lang:", language_reward_func(None, _t))

format: [1.0] | length: [0.5] | correct: [1.0] | lang: [1.0]


Dataset & GRPOTrainer

In [3]:
import inspect
import os
from datasets import load_dataset
from trl import GRPOConfig, GRPOTrainer
from unsloth import is_bfloat16_supported

os.environ["WANDB_PROJECT"] = "PGABL-legal-chatbot-grpo"

# Mengaktifkan kembali variabel SEED yang terhapus karena restart
SEED = 3407

SYSTEM_PROMPT = """Kamu adalah asisten AI hukum yang menjawab dalam Bahasa Indonesia.
Selalu jawab dengan format persis berikut:
<think>
(penalaran langkah demi langkah dalam Bahasa Indonesia, jabarkan dengan beberapa kalimat)
</think>
(jawaban akhir yang ringkas dan jelas dalam Bahasa Indonesia)
Gunakan tag <think> dan </think> masing-masing tepat satu kali, dan jangan menulis apa pun sebelum <think>."""

# Memuat dataset segar langsung dari Hugging Face
raw = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train", download_mode="force_redownload")

print("=== VERIFIKASI ULANG KOLOM DATASET SEGAR ===")
print("Kolom terdeteksi aktif:", raw.column_names)

# Deteksi nama kolom secara aman berdasarkan data baru yang bersih
inst_key = "Instruction" if "Instruction" in raw.column_names else ("instruction" if "instruction" in raw.column_names else "input")
input_key = "Input" if "Input" in raw.column_names else ("input" if "input" in raw.column_names else "input")
out_key = "Output" if "Output" in raw.column_names else ("output" if "output" in raw.column_names else "output")

# Buang sampel kosong atau terlalu panjang agar aman dari OOM
raw = raw.filter(lambda x: x.get(out_key) is not None and x.get(inst_key) is not None and 40 <= len(str(x[out_key])) <= 600 and len(str(x[inst_key])) <= 300)
raw = raw.shuffle(seed=SEED).select(range(1000))

def build_prompt(ex):
    instruction = str(ex.get(inst_key) or "").strip()
    extra = str(ex.get(input_key) or "").strip()
    
    if instruction == extra:
        user_msg = instruction
    else:
        user_msg = instruction if not extra else f"{instruction}\n\n{extra}"
        
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        "answer": str(ex.get(out_key) or "").strip(),
    }

grpo_dataset = raw.map(build_prompt, remove_columns=raw.column_names)
print("Jumlah prompt GRPO sukses terbuat:", len(grpo_dataset))

cfg = dict(
    output_dir="outputs_grpo",
    run_name="grpo-legal-id-v1",
    learning_rate=1e-5,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    max_grad_norm=0.1,
    logging_steps=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # 16 completion/step = 4 prompt x 4 generations
    num_generations=4,                  # kecil agar hemat VRAM
    max_prompt_length=256,
    max_completion_length=384,          # cukup untuk <think> + jawaban, aman dari OOM
    temperature=0.9,
    max_steps=200,
    save_steps=100,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    report_to="wandb",
    seed=SEED,
)

valid = set(inspect.signature(GRPOConfig.__init__).parameters)
training_args = GRPOConfig(**{k: v for k, v in cfg.items() if k in valid})

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward_func, reasoning_length_reward, correctness_reward, language_reward_func],
    args=training_args,
    train_dataset=grpo_dataset,
)

torch.cuda.empty_cache()
trainer.train()
print(f"Peak VRAM: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")
wandb.finish()


README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv: reconstructing file:   0%|          |  0.00B / 41.4MB            

alpaca-gpt4-indonesia.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

=== VERIFIKASI ULANG KOLOM DATASET SEGAR ===
Kolom terdeteksi aktif: ['Unnamed: 0', 'input', 'output']


...

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
1,-0.035973,0.718750,0.515388,138.937500,25.000000,381.000000,0.000000,138.937500,25.000000,381.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.718750,0.515388
2,-0.122770,0.893750,0.356780,62.750000,12.000000,348.000000,0.000000,62.750000,12.000000,348.000000,0.000000,0.018750,0.075000,0.000000,0.000000,0.000000,0.000000,0.875000,0.341565
3,-0.047175,0.750000,0.447214,90.625000,4.000000,309.000000,0.000000,90.625000,4.000000,309.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.750000,0.447214
4,0.057459,0.875000,0.341565,67.812500,16.000000,130.000000,0.000000,67.812500,16.000000,130.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.875000,0.341565
5,0.001548,0.937500,0.250000,40.375000,16.000000,119.000000,0.000000,40.375000,16.000000,119.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.937500,0.250000
6,0.063696,0.750000,0.447214,71.125000,12.000000,270.000000,0.000000,71.125000,12.000000,270.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.750000,0.447214
7,0.373461,1.031250,0.740917,80.562500,7.000000,384.000000,0.062500,60.333336,7.000000,185.000000,0.000000,0.062500,0.250000,0.031250,0.125000,0.062500,0.250000,0.875000,0.341565
8,0.441985,0.937500,0.250000,71.750000,13.000000,384.000000,0.062500,50.933334,13.000000,86.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.937500,0.250000
9,-0.196285,0.812500,0.403113,164.312500,16.000000,384.000000,0.062500,149.666672,16.000000,346.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.812500,0.403113
10,0.000473,0.937500,0.250000,66.000000,21.000000,185.000000,0.000000,66.000000,21.000000,185.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.937500,0.250000


Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Peak VRAM: 9.56 GB


profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▂▁▄▄▁▂▂▁▄▂▂▂▃▃▂▃▃▁▁▃▅▄▄▃▃█▅▄▄▆█▅▃▃▆▇█▆▆▅
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,█▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁██▁▁▁█▁▁█▁▁▁▁▁▁▇▁▁▁██▁
profiling/Time taken: UnslothGRPOTrainer.correctness_reward,▁▂▁▁▁▂▂▁▁▂▁▁▂▁▂▂▁▂▄▄▄▃▅▃▄▇▅▄▇▆▆▅▆▆▄▄▅▅█▃
profiling/Time taken: UnslothGRPOTrainer.format_reward_func,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▃▁▃▃▃▂▆▂▅█▄▆▃▄▃▃▃▅▃▄▅▇▄▃▄
profiling/Time taken: UnslothGRPOTrainer.language_reward_func,▃▄▃▄▅▂▄▁▇▅▄▁▂▆▅▃▅▅▅▇▄▃▃▇▄▃▆▆▂▃▃▂▂▅▂▃▃█▃▄
profiling/Time taken: UnslothGRPOTrainer.reasoning_length_reward,▂▂▂▂▁▂▂▁▂▁▂▁▂▁▁▁▁▂▂▂▃▃▅▄▆▆▅▅▇▄▄▆▅▅█▅▇▅▆█
profiling/Time taken: UnslothGRPOTrainer.transformers.generate,▁▄█▃█▃▄▃▅▃▃██▂▄▇▃▃█▆▆██████████▃█▆██████
train/clip_ratio/high_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+29,...


Save & Push GRPO ke Hugging Face

In [4]:
GRPO_REPO = "putraharifin/llama-3.2-3b-legal-id-GRPO-PGABL-putraharifin25"  # repo baru, jangan timpa model SFT

model.push_to_hub_merged(
    GRPO_REPO,
    tokenizer,
    save_method="merged_16bit",
    token=os.environ["HF_TOKEN"],
    private=False,   # WAJIB public
)

with open("link_huggingface.txt", "w") as f:
    f.write(f"SFT  : https://huggingface.co/{SFT_REPO}\n")
    f.write(f"GRPO : https://huggingface.co/{GRPO_REPO}\n")
print(open("link_huggingface.txt").read())

Unsloth: Restored added_tokens_decoder metadata in putraharifin/llama-3.2-3b-legal-id-GRPO-PGABL-putraharifin25/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


...



Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:35<01:35, 95.13s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:02<00:00, 61.50s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/putraharifin/llama-3.2-3b-legal-id-GRPO-PGABL-putraharifin25`
SFT  : https://huggingface.co/putraharifin/llama-3.2-3b-legal-id-PGABL-putraharifin25
GRPO : https://huggingface.co/putraharifin/llama-3.2-3b-legal-id-GRPO-PGABL-putraharifin25



Uji Inferensi Test Case Wajib

In [5]:
FastLanguageModel.for_inference(model)

test_prompt = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_prompt},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to("cuda")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

raw_output = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=" * 25, "RAW OUTPUT", "=" * 25)
print(raw_output)
print("=" * 62)

# Bukti struktur reasoning + skor reward
has_think = bool(re.search(r"<think>.+?</think>", raw_output, flags=re.DOTALL))
final_answer = get_final_answer(raw_output)
print("Ada blok <think>...</think> :", has_think)
print("Jawaban final               :", final_answer)
print("Skor format / length / lang :",
      format_reward_func(None, [raw_output]),
      reasoning_length_reward(None, [raw_output]),
      language_reward_func(None, [raw_output]))

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


========================= RAW OUTPUT =========================
<think>
Sebagai staf admin, Anda mungkin berhak mendapatkan uang lembur jika Anda bekerja di tempat kerja yang memungkinkan Anda untuk mendapatkan uang lembur. Namun, hal ini tergantung pada perjanjian kerja dan aturan perusahaan Anda. Anda harus memeriksa perjanjian kerja Anda untuk mengetahui apakah Anda berhak mendapatkan uang lembur. Jika Anda bekerja di tempat kerja yang memungkinkan Anda mendapatkan uang lembur, Anda seharusnya menerima uang lembur Anda setelah bekerja selama 3 jam lembur.
</think>
Jadi, Anda berhak mendapatkan uang lembur jika Anda bekerja di tempat kerja yang memungkinkan Anda mendapatkan uang lembur.
Ada blok <think>...</think> : True
Jawaban final               : Jadi, Anda berhak mendapatkan uang lembur jika Anda bekerja di tempat kerja yang memungkinkan Anda mendapatkan uang lembur.
Skor format / length / lang : [1.0] [1.0] [1.0]
